In [1]:
import numpy as np
import tensorflow as tf

In [2]:
data = np.loadtxt('./data.csv', delimiter=',', unpack=True, dtype='float32')
x_data = np.transpose(data[0:2])
y_data = np.transpose(data[2:])

In [3]:
global_step = tf.Variable(0, trainable=False, name='global_step')

X = tf.placeholder(tf.float32, [None, 2])
Y = tf.placeholder(tf.float32, [None, 3])

W1 = tf.Variable(tf.random_uniform([2, 10], -1., 1.))
W2 = tf.Variable(tf.random_uniform([10, 20], -1., 1.))
W3 = tf.Variable(tf.random_uniform([20, 3], -1., 1.))

tf.summary.histogram('weights_1', W1)
tf.summary.histogram('weights_2', W2)
tf.summary.histogram('weights_3', W3)

<tf.Tensor 'weights_3:0' shape=() dtype=string>

In [4]:
with tf.name_scope('layer1'):
    L1 = tf.nn.relu(tf.matmul(X, W1))

with tf.name_scope('layer2'):
    L2 = tf.nn.relu(tf.matmul(L1, W2))

with tf.name_scope('output'):
    model = tf.matmul(L2, W3)

In [5]:
with tf.name_scope('cost'):
    cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=Y, logits=model))
    tf.summary.scalar('cost', cost)

with tf.name_scope('train'):
    train_op = tf.train.AdamOptimizer(learning_rate=0.01).minimize(cost, global_step=global_step)

In [6]:
sess = tf.Session()
saver = tf.train.Saver(tf.global_variables())

In [7]:
ckpt = tf.train.get_checkpoint_state('./model')

if ckpt and tf.train.checkpoint_exists(ckpt.model_checkpoint_path):
    saver.restore(sess, ckpt.model_checkpoint_path)
else:
    sess.run(tf.global_variables_initializer())

INFO:tensorflow:Restoring parameters from ./model\dnn.ckpt-100


In [8]:
merged = tf.summary.merge_all()
writer = tf.summary.FileWriter('./logs', sess.graph)

In [9]:
for step in range(100):
    sess.run(train_op, feed_dict={X: x_data, Y: y_data})
    
    print('Step: %d' % sess.run(global_step), 'Cost: %.3f' % sess.run(cost, feed_dict={X: x_data, Y: y_data}))
    
    summary = sess.run(merged, feed_dict={X: x_data, Y: y_data})
    writer.add_summary(summary, global_step=sess.run(global_step))

Step: 101 Cost: 0.550
Step: 102 Cost: 0.550
Step: 103 Cost: 0.550
Step: 104 Cost: 0.550
Step: 105 Cost: 0.550
Step: 106 Cost: 0.550
Step: 107 Cost: 0.550
Step: 108 Cost: 0.550
Step: 109 Cost: 0.550
Step: 110 Cost: 0.550
Step: 111 Cost: 0.550
Step: 112 Cost: 0.550
Step: 113 Cost: 0.550
Step: 114 Cost: 0.550
Step: 115 Cost: 0.550
Step: 116 Cost: 0.550
Step: 117 Cost: 0.550
Step: 118 Cost: 0.550
Step: 119 Cost: 0.550
Step: 120 Cost: 0.550
Step: 121 Cost: 0.550
Step: 122 Cost: 0.550
Step: 123 Cost: 0.550
Step: 124 Cost: 0.550
Step: 125 Cost: 0.550
Step: 126 Cost: 0.550
Step: 127 Cost: 0.550
Step: 128 Cost: 0.550
Step: 129 Cost: 0.550
Step: 130 Cost: 0.550
Step: 131 Cost: 0.550
Step: 132 Cost: 0.550
Step: 133 Cost: 0.550
Step: 134 Cost: 0.550
Step: 135 Cost: 0.550
Step: 136 Cost: 0.550
Step: 137 Cost: 0.550
Step: 138 Cost: 0.550
Step: 139 Cost: 0.550
Step: 140 Cost: 0.550
Step: 141 Cost: 0.550
Step: 142 Cost: 0.550
Step: 143 Cost: 0.550
Step: 144 Cost: 0.550
Step: 145 Cost: 0.550
Step: 146 

In [10]:
saver.save(sess, './model/dnn.ckpt', global_step=global_step)

'./model/dnn.ckpt-200'

In [11]:
prediction = tf.argmax(model, 1)
target = tf.argmax(Y, 1)

print('Prediction:', sess.run(prediction, feed_dict={X: x_data}))
print('Target:', sess.run(target, feed_dict={Y: y_data}))

Prediction: [0 1 2 0 0 2]
Target: [0 1 2 0 0 2]


In [12]:
is_correct = tf.equal(prediction, target)
accuracy = tf.reduce_mean(tf.cast(is_correct, tf.float32))

print('Accuracy: %2.f' % sess.run(accuracy * 100, feed_dict={X: x_data, Y: y_data}))

Accuracy: 100
